# Readme E-Rara

Dieses Jupyter Notebook bereitet Datenobjekte und Metadaten aus dem E-Rara-Bestand vor für die digitale Langzeitarchivierung (DLZA) der ZHB Luzern, basierend auf der GOCFL implementierung von Jürgen Enge, https://github.com/je4/gocfl .

Um eine Collection korrekt vorzubereiten, sollten die einzelnen Scripts immer alle in der richtigen Reihenfolge ausgeführt werden. 

## 1 - Orderstruktur erstellen

Die Scripts in diesem Notebook basieren darauf, dass folgende Ordnerstruktur vorhanden ist, und dass die Ordner jeweils zu Beginn geleert werden. 

- Ordner 'files': hier werden diverse Dateien (z.B. eine Textdatei mit allen Signaturen) abgelegt
- Ordner 'info': hier werden die info.json Dateien abgelegt.
- Ordner 'metadata': hier werden die semantischen Metadaten zu den LZA-Objekten abgelegt
- Ordner 'objects': hier werden die zu archivierenden Datenobjekte (Payload) abgelegt. 

In [ ]:
import os
import shutil
from datetime import datetime

# remove existing directories
if os.path.exists('files'):    
    shutil.rmtree('files')

if os.path.exists('info'):
    shutil.rmtree('info')
    
if os.path.exists('metadata'):
    shutil.rmtree('metadata')

# exception: don't remove directory 'objects'
if os.path.exists('objects'):
    pass
else:
    os.mkdir('objects')
 
    
# create necessary directories:
os.mkdir('files')
os.mkdir('info')
os.mkdir('metadata')  

# list all directories in current working directory:
working_directory = './'
print(f"Working directory contains the following directories:")
items = os.listdir(working_directory)
for item in items:
    if os.path.isdir(item):
        print(item)
        
        
print("Finished at ",datetime.today().strftime('%Y-%m-%d %H:%M:%S'))

## Basiskonfiguration info.json

Dieses Script erstellt eine info.json Datei für alle Objekte der Collection und legt sie im Ordner 'info' als JSON Datei ab. 

### Config.py

Beispieldaten für ZHB E-Rara. Anpassungen können in der config.py vorgenommen werden. 

    user = 'name of person ingesting this object'
    address = 'mailto:someone@internet.com'
    collection = 'ZHB E-Rara'
    collection_id = 'zhb_erara'
    last_changed = 'yyyy-mm-dd'
    organisation = 'Zentral- und Hochschulbibliothek Luzern'
    organisation_id = 'zhb'
    signature = 'zhb_'
    
Die E-Rara-Signaturen setzen sich aus dem E-Rara DOI zusammen. Im Config-File wird die Abteilung hinzugefügt, das Script ergänzt den DOI.     
    
### OAI configuration 

Für die E-Rara-Bestände wird die Zenodo-OAI-Schnittstelle verwendet, da hier die Original-ZIP-Kapseln liegen und auf alle andern Systeme verlinkt wird (Alma, E-Rara).

Zenodo: siehe https://developers.zenodo.org/#oai-pmh

    community: lara_e-rara
    set name for OAI listrecords: user-lara_e-rara

### Export info.json

Für jeden einzelnen record wird eine info.json-Datei erstellt im Format info/signature.json.
Das ganze Set wird am Ende noch als json- und Excel-Datei exportiert ins directory 'files' als menschenlesbarer Nachweis, welche Datenobjekte eingelagert wurden. Auch eine Liste aller Signaturen wird als Text-Datei dort abgelegt. 


In [ ]:
from sickle import Sickle
import json
import config
import requests
import pandas as pd
from datetime import datetime

# Initialize the client by passing the base URL and fetch records from the OAI Set

base_url = 'https://zenodo.org/oai2d'
prefix = 'oai_dc'
set_name = 'user-lara_e-rara'

sickle = Sickle(base_url)
records = sickle.ListRecords(metadataPrefix=prefix, set=set_name)
record = records.next()

completed_iterating = False
recordCount = 1

# other output variables

completeSet = []    
today = datetime.today().strftime('%Y-%m-%d')
sigfile = "files/signatures.txt"

# Iterate through records and collect metadata info for json export

# !!!! Uncomment one of the next 2 lines for testing/production:

#while not completed_iterating:
while recordCount < 5:
    try:
        
        infoSet = {}                
        infoSet["additional"] = ''
        infoSet["address"] = config.address
        infoSet["collection"] = config.collection
        infoSet["collection_id"] = config.collection_id
        infoSet["created"] = record.metadata["date"][0]
        infoSet["identifiers"] = record.metadata["identifier"]
        infoSet["ingest_workflow"] = config.ingest_workflow
        infoSet["keywords"] = config.keywords
        infoSet["last_changed"] = today
        infoSet["organisation"] = config.organisation
        infoSet["organisation_id"] = config.organisation_id
        infoSet["references"] = record.metadata["relation"]
        infoSet["sets"] = config.sets
        infoSet["signature"] = ''
        infoSet["title"] = record.metadata["title"][0]
        infoSet["user"] = config.user                # print(infoSet["references"])
        
        # get e-rara doi, alma id, zenodo id:
        erara_doi = ''
        alma_id = ''
        zenodo_id = ''
        
        for i in infoSet["references"]:
            if i.startswith('doi:10.3931/e-rara'):
                erara_doi = i[4:]         
            
            if i.startswith('url:https://rzs') or i.startswith('url:https://swisscovery'):                
                alma_id = i.partition('alma')[2]  
               
            if i.startswith('doi:10.5281/zenodo.'):
                zenodo_id = i[19:]
        
        # save identifiers: 
        
        infoSet["identifiers"].insert(0, erara_doi)
        infoSet["identifiers"].insert(0, str(zenodo_id))
        infoSet["identifiers"].insert(0, str(alma_id))                 
         
        print("E-Rara-DOI:", erara_doi, "| MMS-ID:", alma_id, "| Zenodo ID:", zenodo_id)  
        
         # save beginning of file name to 'additional' to later create object folders:
        object_folder = erara_doi.replace('.','_').replace('/','_') 
        infoSet["additional"] = object_folder                   
        
        #create signature:
        signature = config.signature+object_folder
        infoSet["signature"] = signature
        print("Signature:",signature)
        #debugging:        print(infoSet)
        
        # prepare filename for json export
        info_json = json.dumps(infoSet, indent=4, ensure_ascii=False)
        infofile = f"info/{signature}.json"
        with open(infofile, "w") as outfile:
            outfile.write(info_json)
            print(f"---\ninfo.json saved as {infofile}")
        
        # add info to completeSet
        completeSet.append(infoSet)             
            
        # append signature to signatures.txt
        with open(sigfile, 'a') as file:
            file.write(signature)
            file.write("\n")
            print(f"Signature {signature} appended to {sigfile}\n")

        #continue with next record
        recordCount = recordCount +1
        record = records.next()
        
    except StopIteration:
        completed_iterating = True

# Writing completeSet as json file
fulldump = json.dumps(completeSet, indent=4, ensure_ascii=False)
fulljsonfile = "files/erara_complete_set.json"
with open(fulljsonfile, "w") as outfile:
    outfile.write(fulldump)
    print(f"---\nJSON File for entire collection written to {fulljsonfile}")
    
# Writing completeSet as Excel file
fullexcelfile = "files/erara_complete_set.xlsx"
df_json = pd.read_json(fulljsonfile)
df_json.to_excel(fullexcelfile)
print(f"Excel File for entire collection written to {fullexcelfile}")
print("Finished at ",datetime.today().strftime('%Y-%m-%d %H:%M:%S'))


## Semantische Metadaten

### Marcxml aus Alma (SRU)

Mit der alma_id werden die MARC-Daten via SRU aus Alma extrahiert und abgespeichert unter metadata/{signature}/signature.xml. Für gocfl create müssen die Metadaten pro Objekt in einem eigenen Ordner liegen. 
Grundsätzlich könnten noch weitere Metadaten in diesem Ordner abgelegt werden, dazu wird jedoch der jeweilige Identifier benötigt (welcher z.B. für E-Rara nicht bekannt ist).     

In [ ]:
import requests
import json
import os
from datetime import datetime


url = "https://slsp-rzs.alma.exlibrisgroup.com/view/sru/41SLSP_RZS"
operation = '?version=1.2&operation=searchRetrieve&recordSchema=marcxml&query=rec.id='
sru = url+operation

file_name = 'files/erara_complete_set.json'

with open(file_name) as data_file:    
    data = json.load(data_file)
    for value in data:
        # find necessary values
        alma_id = value["identifiers"][0]
        foldername = value["additional"]
        signature = value["signature"]
        # get SRU response
        query = sru+alma_id
        response = requests.get(query)
        if response.status_code != 200:
            raise Exception(f"SRU request failed with status code {response.status_code}")
        
        # Save the response content as xml to a new directory
        os.mkdir(f'metadata/{foldername}')
        metafile = f"metadata/{foldername}/{signature}.xml"
        
        with open(metafile, 'wb') as file:
            file.write(response.content)
            print(f"\nRecord with ID {alma_id} saved as {metafile}\n---")

print("Finished at ",datetime.today().strftime('%Y-%m-%d %H:%M:%S'))


### Abholung Datenobjekte 
Die E-Rara-Zipkapseln der ZHB liegen auf LARA (Zenodo). 
Sie sind nach folgender Struktur benannt:
DOI (Punkt/Schrägstrich ersetzt durch Unterstrich) _ (Digitalisierungsdatum/Workflow?) _ master _ version . zip

Beispiel:

     10_3931_e-rara-86416_20201027T114533_master_ver1.zip

Für gocfl wird vorausgesetzt, dass im Ordner 'objects' ein Unterordner mit dem DOI existiert, worin die ZIP-Kapsel liegt (in wenigen Fällen auch mehrere Zip-Kapseln). 

Der DOI / Foldername ist in dieser Form (mit Unterstrichen) im Feld "additional" in der JSON-Datei gespeichert. 


In [1]:
import requests
import json
import os
from zipfile import ZipFile
from datetime import datetime

file_name = 'files/erara_complete_set.json'

with open(file_name) as data_file:    
    data = json.load(data_file)
    for value in data:
        # find necessary values
        zenodo_id = value["identifiers"][1]
        foldername = value["additional"]
        signature = value["signature"]
        
        # make a directory for each object
        
        if os.path.exists(f'objects/{foldername}'):
            pass
        else:
            os.mkdir(f'objects/{foldername}')
                  
        # get the file download link from zenodo:        
        zenodo_link = f'https://zenodo.org/api/records/{zenodo_id}'        #print(zenodo_link)
        response = requests.get(zenodo_link)
        zenodo_object = response.json()
        for files in zenodo_object['files']:
            download_url = files['links']['self']
            local_file = download_url.split("/")[-1]              
            local_file = f'objects/{foldername}/{local_file}'
            
            with ZipFile(local_file, 'w') as zip_object:
                print(f'Zip created: {local_file}')
            response = requests.get(download_url, stream = True)
            print("Response:", response.status_code)
            
            with open(local_file, 'wb') as f:
                c = 1
                for chunk in response.iter_content(chunk_size = 10_000_000):
                    f.write(chunk)                    
                    print('%d chunks written' % c)
                    c += 1
            

print("Finished at ",datetime.today().strftime('%Y-%m-%d %H:%M:%S'))

Zip created: objects/10_3931_e-rara-88937/10_3931_e-rara-88937_20210810T063306_master_ver1.zip
Response: 200
1 chunks written
2 chunks written
3 chunks written
4 chunks written
5 chunks written
6 chunks written
7 chunks written
8 chunks written
9 chunks written
10 chunks written
11 chunks written
12 chunks written
13 chunks written
14 chunks written
15 chunks written
16 chunks written
17 chunks written
18 chunks written
19 chunks written
20 chunks written
21 chunks written
Zip created: objects/10_3931_e-rara-86391/10_3931_e-rara-86391_20201027T114533_master_ver1.zip
Response: 200
1 chunks written
2 chunks written
3 chunks written
4 chunks written
5 chunks written
6 chunks written
7 chunks written
8 chunks written
9 chunks written
10 chunks written
11 chunks written
12 chunks written
Zip created: objects/10_3931_e-rara-86386/10_3931_e-rara-86386_20201027T114533_master_ver1.zip
Response: 200
1 chunks written
2 chunks written
3 chunks written
4 chunks written
5 chunks written
6 chunks wri